# Volume 8, Chapter 4: Curvature and Gauss-Bonnet (Python companion)

Curvature at work: the angle defect of triangle meshes, shortest routes on the WGS84 ellipsoid, and the spectrum of the flat torus and of its grid discretization.

*Before each solution, predict what the output will look like.*

## Exercises

**P4.1 (★) CS application: curvature of a triangle mesh.** Computer graphics stores surfaces as triangle meshes, which have no second derivatives. The *angle defect* at a vertex $v$ is $\delta(v)=2\pi-\sum(\text{angles of the triangles at }v)$. (a) Prove the discrete Gauss--Bonnet theorem $\sum_v\delta(v)=2\pi(V-E+F)$ for a closed mesh (count angles triangle by triangle, and use $3F=2E$). (b) Check it numerically on a subdivided icosahedron (a sphere) and on a triangulated torus. (c) On the torus, compare $\delta(v)$ divided by one third of the area of the triangles at $v$ with the exact curvature $K=\frac{\cos u}{a(R+a\cos u)}$ of Exercise 4.2, and watch the error as the mesh is refined.

**P4.2 (★★) CS application: shortest routes on the Earth.** The Earth is close to the ellipsoid of revolution $\frac{x^2+y^2}{a^2}+\frac{z^2}{b^2}=1$ with the WGS84 values $a=6378.137$ km, $f=\frac{a-b}a=\frac1{298.257223563}$. (a) For a surface $F=0$ in $\R^3$, a unit-speed curve is a geodesic exactly when its acceleration is normal; show that this gives $x''=-\frac{\langle x',H_F\,x'\rangle}{|\nabla F|^2}\nabla F$ and integrate it with a Runge--Kutta solver. Check that the speed and the Clairaut quantity $r\sin\psi$ (distance to the axis times the sine of the angle with the meridian) are conserved. (b) Compute the length of the quarter meridian, from the equator to the pole, and compare with the value $10\,001.966$ km quoted in geodesy references. (c) Solve the two-point problem by shooting (unknowns: the initial azimuth and the length) for a few pairs of cities, and compare with the great-circle distance on a sphere of radius $6371$ km.

**P4.3 (★★) Hearing the shape of a torus, and Weyl's law.** (a) On the flat torus $\R^2/\Z^2$ (area $1$), list the eigenvalues $4\pi^2n$ of $-\Delta$ with multiplicity $r_2(n)$ (Theorem 4.19) and check Weyl's law $N(\lambda)\sim\frac{\lambda}{4\pi}$, where $N(\lambda)$ counts eigenvalues $\le\lambda$. (b) Discretize the torus as an $m\times m$ periodic grid; its graph Laplacian is the five-point stencil. Find its eigenvalues by the same separation of variables, and compare the lowest ones with $4\pi^2n$. (c) Compare the square torus $\R^2/\Z^2$ with the rectangular torus $\R^2/(\sqrt2\Z\times\frac1{\sqrt2}\Z)$. Both have area $1$, so Weyl's law cannot tell them apart; show that their spectra differ, and explain what the first nonzero eigenvalue measures. (Two flat tori with the same spectrum need not be isometric: Milnor's 1964 example lives in dimension $16$.)

## Solutions

### P4.1

In [ ]:
import numpy as np

def icosphere(levels):
    p = (1 + 5 ** 0.5) / 2
    V = [(-1, p, 0), (1, p, 0), (-1, -p, 0), (1, -p, 0), (0, -1, p), (0, 1, p),
         (0, -1, -p), (0, 1, -p), (p, 0, -1), (p, 0, 1), (-p, 0, -1), (-p, 0, 1)]
    V = [np.array(v, float) / np.linalg.norm(v) for v in V]
    F = [(0, 11, 5), (0, 5, 1), (0, 1, 7), (0, 7, 10), (0, 10, 11), (1, 5, 9), (5, 11, 4), (11, 10, 2),
         (10, 7, 6), (7, 1, 8), (3, 9, 4), (3, 4, 2), (3, 2, 6), (3, 6, 8), (3, 8, 9), (4, 9, 5), (2, 4, 11),
         (6, 2, 10), (8, 6, 7), (9, 8, 1)]
    for _ in range(levels):
        mid, G = {}, []
        def m(i, j):
            key = (min(i, j), max(i, j))
            if key not in mid:
                w = V[i] + V[j]
                V.append(w / np.linalg.norm(w))
                mid[key] = len(V) - 1
            return mid[key]
        for a, b, c in F:
            ab, bc, ca = m(a, b), m(b, c), m(c, a)
            G += [(a, ab, ca), (b, bc, ab), (c, ca, bc), (ab, bc, ca)]
        F = G
    return np.array(V), np.array(F)

def torus(n, R=2.0, a=1.0):
    u, v = np.meshgrid(np.arange(n) * 2 * np.pi / n, np.arange(2 * n) * 2 * np.pi / (2 * n), indexing="ij")
    V = np.stack([(R + a * np.cos(u)) * np.cos(v), (R + a * np.cos(u)) * np.sin(v), a * np.sin(u)], -1).reshape(-1, 3)
    idx = lambda i, j: (i % n) * (2 * n) + (j % (2 * n))
    F = []
    for i in range(n):
        for j in range(2 * n):
            F += [(idx(i, j), idx(i + 1, j), idx(i + 1, j + 1)), (idx(i, j), idx(i + 1, j + 1), idx(i, j + 1))]
    return V, np.array(F), u.ravel()

def defects(V, F):
    ang = np.zeros(len(V)); area = np.zeros(len(V))
    for k in range(3):
        i, j, l = F[:, k], F[:, (k + 1) % 3], F[:, (k + 2) % 3]
        e1, e2 = V[j] - V[i], V[l] - V[i]
        c = np.einsum("ij,ij->i", e1, e2) / (np.linalg.norm(e1, axis=1) * np.linalg.norm(e2, axis=1))
        np.add.at(ang, i, np.arccos(np.clip(c, -1, 1)))
    tri = 0.5 * np.linalg.norm(np.cross(V[F[:, 1]] - V[F[:, 0]], V[F[:, 2]] - V[F[:, 0]]), axis=1)
    for k in range(3):
        np.add.at(area, F[:, k], tri / 3)
    return 2 * np.pi - ang, area

def euler(V, F):
    E = {tuple(sorted(e)) for f in F for e in ((f[0], f[1]), (f[1], f[2]), (f[2], f[0]))}
    return len(V) - len(E) + len(F)

for lev in range(4):
    V, F = icosphere(lev)
    d, A = defects(V, F)
    print(f"icosphere level {lev}: V={len(V):5d}  chi={euler(V,F)}  sum of defects / 2pi = {d.sum()/(2*np.pi):.12f}"
          f"   max |defect/area - 1| = {np.abs(d/A - 1).max():.4f}")
print()
for n in (8, 16, 32, 64):
    V, F, u = torus(n)
    d, A = defects(V, F)
    K = np.cos(u) / (1.0 * (2.0 + np.cos(u)))
    print(f"torus n={n:3d}: chi={euler(V,F)}  sum of defects = {d.sum():+.2e}   max |defect/area - K| = {np.abs(d/A - K).max():.4f}")

(a) Each triangle has angle sum $\pi$, so the sum of all angles at all vertices is $\pi F$, and $\sum_v\delta(v)=2\pi V-\pi F$. Every edge lies on two triangles and every triangle has three edges, so $3F=2E$, that is $F=2E-2F$, and $2\pi V-\pi F=2\pi V-2\pi E+2\pi F=2\pi\chi$. This is an exact identity, true for every mesh however crude: the discrete curvature obeys Gauss--Bonnet on the nose, which is why it is the standard curvature of geometry processing.

(b) The output shows $\sum\delta/2\pi=2$ to twelve digits on all icospheres and $\sum\delta=0$ to rounding on all tori, with $\chi=2$ and $\chi=0$ counted directly.

(c) On the torus, whose mesh is regular (every vertex has six triangles in the same pattern), the error in $K$ is divided by about $4$ each time $n$ doubles: second-order convergence. On the icosphere it is not: the maximal error stalls near $15\%$. The culprits are the twelve original vertices of the icosahedron, which have five neighbours instead of six; however fine the mesh, the defect there is not the right multiple of the area. The integrated statement (a), on the other hand, holds exactly for every mesh. Pointwise discrete curvature is fragile, total curvature is robust: this is a theme of discrete differential geometry.

### P4.2

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve

a = 6378.137
f = 1 / 298.257223563
b = a * (1 - f)
D = np.array([1 / a**2, 1 / a**2, 1 / b**2])       # F(x) = sum D_i x_i^2 - 1, Hessian 2 diag(D)

def rhs(s, Y):
    x, v = Y[:3], Y[3:]
    g = 2 * D * x
    acc = -(2 * np.sum(D * v * v)) / (g @ g) * g
    return np.r_[v, acc]

def point(lat, lon):
    """geodetic latitude/longitude in degrees -> point of the ellipsoid"""
    phi, lam = np.radians(lat), np.radians(lon)
    N = a / np.sqrt(1 - f * (2 - f) * np.sin(phi) ** 2)
    return np.array([N * np.cos(phi) * np.cos(lam), N * np.cos(phi) * np.sin(lam), N * (1 - f) ** 2 * np.sin(phi)])

def frame(x):
    """unit north and east tangent vectors at x"""
    n = D * x; n /= np.linalg.norm(n)
    east = np.cross([0, 0, 1], n); east /= np.linalg.norm(east)
    return np.cross(n, east), east

def shoot(x0, az, L, **kw):
    north, east = frame(x0)
    v0 = np.cos(az) * north + np.sin(az) * east
    return solve_ivp(rhs, (0, L), np.r_[x0, v0], method="DOP853", rtol=1e-12, atol=1e-9, **kw)

# (a) invariants along a long geodesic
x0 = point(10, 0)
sol = shoot(x0, np.radians(50), 30000, dense_output=True)
s = np.linspace(0, 30000, 7)
for si in s:
    Y = sol.sol(si); x, v = Y[:3], Y[3:]
    r = np.hypot(x[0], x[1])
    east = np.array([-x[1], x[0], 0]) / r
    print(f"s={si:7.0f} km  |x'|-1 = {np.linalg.norm(v)-1:+.1e}   F(x) = {np.sum(D*x*x)-1:+.1e}   Clairaut r sin(psi) = {r*(v@east):.9f}")

# (b) quarter meridian: shoot north from the equator until z' = 0 at the pole region; integrate to the pole by event
ev = lambda s, Y: Y[0]            # x = 0 means we passed the pole on the meridian lon = 0
ev.terminal, ev.direction = True, -1
q = shoot(point(0, 0), 0.0, 20000, events=ev)
print(f"\nquarter meridian: {q.t_events[0][0]:.6f} km   (geodesy references: 10001.966 km)")
print(f"on a sphere of radius 6371 km: {np.pi/2*6371:.3f} km")

# (c) two-point problem by shooting
cities = {"Paris": (48.8566, 2.3522), "Tokyo": (35.6762, 139.6503), "Tunis": (36.8065, 10.1815),
          "New York": (40.7128, -74.0060), "Sydney": (-33.8688, 151.2093), "Quito": (-0.1807, -78.4678)}

def sphere_dist(p, q, R=6371.0):
    (la1, lo1), (la2, lo2) = np.radians(p), np.radians(q)
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(h))

def geodesic(p, q):
    x0, x1 = point(*p), point(*q)
    # initial guess from the sphere: initial bearing and distance
    (la1, lo1), (la2, lo2) = np.radians(p), np.radians(q)
    az0 = np.arctan2(np.sin(lo2 - lo1) * np.cos(la2), np.cos(la1) * np.sin(la2) - np.sin(la1) * np.cos(la2) * np.cos(lo2 - lo1))
    L0 = sphere_dist(p, q)
    north, east = frame(x1)
    def miss(z):
        az, L = z
        y = shoot(x0, az, L).y[:3, -1] - x1
        return [y @ north, y @ east]
    az, L = fsolve(miss, [az0, L0], xtol=1e-13)
    end = shoot(x0, az, L).y[:3, -1]
    return L, np.degrees(az), np.linalg.norm(end - x1)

pairs = [("Paris", "Tokyo"), ("Paris", "New York"), ("Tunis", "Sydney"), ("Quito", "Tokyo"), ("Paris", "Tunis")]
print(f"\n{'route':20s} {'ellipsoid':>11s} {'sphere':>11s} {'diff %':>8s}  azimuth  miss")
for A, B in pairs:
    L, az, err = geodesic(cities[A], cities[B])
    S = sphere_dist(cities[A], cities[B])
    print(f"{A+' - '+B:20s} {L:11.3f} {S:11.3f} {100*(S-L)/L:+8.3f}  {az:7.2f}  {err:.1e} km")

(a) The geodesic condition says that $x''$ is a multiple of $\nabla F$. Differentiating $F(x(s))=0$ twice gives $\langle\nabla F,x''\rangle+\langle x',H_Fx'\rangle=0$, which fixes the multiple: $x''=-\frac{\langle x',H_Fx'\rangle}{|\nabla F|^2}\nabla F$. The printout shows that along $30\,000$ km the speed stays $1$ and the point stays on the ellipsoid to about $10^{-11}$, and that $r\sin\psi$ is constant to many digits. Clairaut's relation is the conservation law coming from the rotational symmetry (the rotations about the axis are isometries), in the same way as angular momentum.

(b) The integration gives the quarter meridian as $10\,001.966$ km to the metre, the value in geodesy references; on the sphere of radius $6371$ km one gets $10\,007.5$ km. The original metre was defined in 1791 as one ten-millionth of this quarter meridian, which is why the answer is so close to $10\,000$ km: the residual $2$ km comes from errors in the survey of Delambre and Méchain and from the later choice of a better ellipsoid.

(c) Shooting converges in a few Newton steps from the spherical guess, and the end point lands within a micrometre of the target. The sphere of radius $6371$ km is off by up to about a quarter of a percent, which is $24$ km on Paris--Tokyo and $16$ km on Paris--New York; the sign and size of the error depend on the latitudes and directions involved, so no single radius fixes it. Navigation software uses the ellipsoid; Karney's algorithms (2013) solve exactly this two-point problem with series expansions instead of a general ODE solver, to nanometre accuracy.

### P4.3

In [ ]:
import numpy as np

# (a) spectrum of the flat torus R^2/Z^2
M = 60
k = np.arange(-M, M + 1)
n = (k[:, None] ** 2 + k[None, :] ** 2).ravel()
lam = np.sort(4 * np.pi ** 2 * n)
print("lowest eigenvalues / 4pi^2 with multiplicity:")
vals, mult = np.unique(n[n <= 10], return_counts=True)
print("  ", dict(zip(vals.tolist(), mult.tolist())))
for L in (1e3, 1e4, 1e5):
    N = np.sum(lam <= L)
    print(f"N({L:.0e}) = {N:6d}   Weyl lambda/(4 pi) = {L/(4*np.pi):9.1f}   ratio = {N/(L/(4*np.pi)):.4f}")

# (b) the m x m periodic grid with spacing h = 1/m: eigenvalues of the five-point Laplacian
for m in (16, 64, 256):
    h = 1 / m
    j = np.arange(m)
    mu1 = (4 / h**2) * np.sin(np.pi * j / m) ** 2         # one direction
    grid = np.sort((mu1[:, None] + mu1[None, :]).ravel())
    print(f"m={m:3d}: lowest grid eigenvalues / 4pi^2 =", np.round(grid[:10] / (4 * np.pi ** 2), 4))

# (c) rectangular tori R^2/(aZ x bZ): eigenvalues 4pi^2 (j^2/a^2 + k^2/b^2)
def spec(a, b, cnt=10):
    j = np.arange(-10, 11)
    e = (j[:, None] ** 2 / a**2 + j[None, :] ** 2 / b**2).ravel()
    return np.round(np.sort(e)[:cnt], 3)
print("\neigenvalues / 4pi^2, both tori of area 1")
print("square 1 x 1             :", spec(1, 1))
print("rectangle sqrt2 x 1/sqrt2:", spec(np.sqrt(2), 1 / np.sqrt(2)))
for L in (1e4, 1e5):
    j = np.arange(-400, 401)
    e = 4 * np.pi**2 * (j[:, None] ** 2 / 2 + 2 * j[None, :] ** 2)
    print(f"rectangle: N({L:.0e}) / (lambda/4pi) = {np.sum(e <= L)/(L/(4*np.pi)):.4f}")

(a) The multiplicities $1,4,4,0,4,8,0,0,4,4,8$ for $n=0,\dots,10$ are the values of $r_2(n)$ (they vanish at $n=3,6,7$, which are not sums of two squares). $N(\lambda)$ counts lattice points in a disc of radius $\frac{\sqrt\lambda}{2\pi}$, so $N(\lambda)\approx\pi\frac{\lambda}{4\pi^2}=\frac\lambda{4\pi}$: Weyl's law here is Gauss's circle problem, and the ratio tends to $1$ as printed.

(b) Separation of variables works on the grid exactly as on the torus: $e^{2\pi i(jx+ky)}$ sampled on the grid is an eigenvector, with eigenvalue $\frac4{h^2}\bigl(\sin^2\frac{\pi j}m+\sin^2\frac{\pi k}m\bigr)$. Since $\frac4{h^2}\sin^2\frac{\pi j}m=4\pi^2j^2\bigl(1+O(\frac{j^2}{m^2})\bigr)$, the low eigenvalues converge to $4\pi^2n$ with the same multiplicities; the high ones (with $j$ near $m$) do not, which is why graph Laplacians reproduce only the bottom of the spectrum. This is the basis of spectral methods in graphics and of spectral clustering, where the lowest eigenvectors of a graph Laplacian are used as coordinates.

(c) The rectangle has first nonzero eigenvalue $4\pi^2\cdot\frac12$, with multiplicity $2$ (the functions $e^{\pm2\pi ix/\sqrt2}$ along the long side), against $4\pi^2$ with multiplicity $4$ for the square, while the counting functions agree asymptotically, as Weyl's law predicts for equal areas. On a flat torus $\R^2/\Lambda$ the eigenvalues are $4\pi^2|w|^2$ for $w$ in the dual lattice, and the first one is $4\pi^2/\ell^2$, where $\ell$ is the distance between neighbouring parallel lines of $\Lambda$ in the slowest direction: the longer a torus is in some direction, the lower its fundamental tone, exactly as a longer string sounds lower.